In [1]:

import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.document_loaders import CSVLoader,TextLoader, PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from uuid import uuid4

from pydantic import BaseModel, Field
from typing import Optional, TypedDict, Literal

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import pandas as pd
from pprint import pprint



C:\Users\User\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')
os.environ['PINECONE_API_KEY']=os.getenv('PINECONE_API_KEY')
os.environ['LANGCHAIN_API_KEY']=os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN PROJECT']=os.getenv('LANGCHAIN_PROJECT')

# for tracing and monitoring purposes
os.environ['LANGCHAIN_TRACING_V2']="true"  # this is as per Langchain documentation

In [55]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)
llm_model=ChatGroq(model="llama-3.1-8b-instant",temperature=0.5)

In [4]:
col_metadata=CSVLoader('columns_agent.csv')

col_docs=col_metadata.load()


col_id=[]
cols=[]

for doc in col_docs:
    each_col=doc.page_content.split("\n")
    col_id.append(each_col[0].split(":")[1].strip())
    cols.append(each_col[1].split(":")[1].strip())

print(col_id,cols)

['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS'] ['Client number', 'Gender', 'Is there a car', 'Is there a property', 'Number of children', 'Annual income', 'Income category', 'Education level', 'Marital status', 'Way of living', 'Birthday', 'Start date of employment', 'Is there a mobile phone', 'Is there a work phone', 'Is there a phone', 'Is there an email', 'Occupation', 'Family size']


In [5]:
pc_apikey=os.getenv('PINECONE_API_KEY')
pc=Pinecone(api_key=pc_apikey)

indexname="dataassitant"

if not pc.has_index(indexname):
    pc.create_index(
        name=indexname,
        dimension=len(hf_embeddings.embed_query("Hey!")),
        metric="cosine",
        spec=ServerlessSpec(cloud='aws',region="us-east-1")  # ServerlessSpec is a configuration object used in Pinecone to define how and where a serverless index should be deployed. It’s part of the Pinecone SDK and is especially relevant when you're creating indexes that don’t rely on dedicated pods, making them more scalable and cost-efficient

    )

index=pc.Index(indexname)

pc_vectors=PineconeVectorStore(index=index,embedding=hf_embeddings)

col_docs_with_metadata=[]
for doc in col_docs : 
   each_col=doc.page_content.split("\n")
   col_docs_with_metadata.append(Document(page_content=each_col[1].split(":")[1].strip(),metadata={"id":each_col[0].split(":")[1].strip()}))

col_docs_with_metadata
    
pc_vectors.add_documents(documents=col_docs_with_metadata,ids=col_id)

['ID',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'FLAG_MOBIL',
 'FLAG_WORK_PHONE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS']

In [6]:
pc_vectors.similarity_search("What is age?")
retreiver = pc_vectors.as_retriever(search_kwargs={'k': 1})
retreiver.invoke("What is Age and income?")

[Document(id='NAME_INCOME_TYPE', metadata={'id': 'NAME_INCOME_TYPE'}, page_content='Income category')]

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000021046F08E10>: Failed to resolve \'api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: 650
API Key: lsv2_********************************************7atrace=b27ffd0b-907c-405a-95e6-f0827798437a,id=b27ffd0b-907c-405a-95e6-f0827798437a
Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /

In [7]:
class llmResponse(BaseModel):
    Valid:Literal["yes","no"]=Field(description="If the description can be used to answer the question then Yes else No")
    Reason:str=Field(description="your reasoning behind the decision of Valid field")

Output_parser=PydanticOutputParser(pydantic_object=llmResponse)

In [8]:
class topicClassifier(BaseModel):
    topics:list[str]=Field(description="Extract broad topics based on the question by removing unnecessary words and return the topics as a LIST ONLY")
topics_parser=PydanticOutputParser(pydantic_object=topicClassifier)

In [9]:
class analysisTechniqueGenerator(BaseModel):
    analysis :str=Field(description="Generate idea for the analysis")

analysis_parser=PydanticOutputParser(pydantic_object=analysisTechniqueGenerator)

In [10]:
topicPrompt=""" 
            You are a smart AI Assistant with strong NLP capabilities, you will extract core specific topics from the {question} which can be used to find the database columns and strip out unnecessary words
            Return the result in the format that matches this schema :\n{format_instructions}

           """

topic_prompt_template=ChatPromptTemplate.from_template(topicPrompt)

topics_llm_model=topic_prompt_template|llm_model|topics_parser

try:
    list_of_topics=topics_llm_model.invoke(
                                            {
                                            'question':" I want to analyze the number of millenial customers who have alteast master degree with family of 4 people and earns more than 100K yearly",
                                            'format_instructions':topics_parser.get_format_instructions()
                                            }
                                        )
except Exception as e:
    print("!!!!Error Occured!!!!")
    print(e)

list_of_topics.topics

final_documents_list=[]
for eachTopic in list_of_topics.topics:
    print(eachTopic+"--->")
    print(retreiver.invoke(eachTopic),"\n")
    final_documents_list.append(retreiver.invoke(eachTopic)[0])
    
final_documents_list


unique_documents={}
for d in final_documents_list:
    if d.id in unique_documents:
        pass
    else:
        unique_documents[d.id]=d.page_content
        
unique_documents

Millennials--->
[Document(id='DAYS_BIRTH', metadata={'id': 'DAYS_BIRTH'}, page_content='Birthday')] 

Customers--->
[Document(id='OCCUPATION_TYPE', metadata={'id': 'OCCUPATION_TYPE'}, page_content='Occupation')] 

Master's Degree--->
[Document(id='NAME_EDUCATION_TYPE', metadata={'id': 'NAME_EDUCATION_TYPE'}, page_content='Education level')] 

Family--->
[Document(id='CNT_FAM_MEMBERS', metadata={'id': 'CNT_FAM_MEMBERS'}, page_content='Family size')] 

Income--->
[Document(id='AMT_INCOME_TOTAL', metadata={'id': 'AMT_INCOME_TOTAL'}, page_content='Annual income')] 

Earnings--->
[Document(id='AMT_INCOME_TOTAL', metadata={'id': 'AMT_INCOME_TOTAL'}, page_content='Annual income')] 

100K--->
[Document(id='AMT_INCOME_TOTAL', metadata={'id': 'AMT_INCOME_TOTAL'}, page_content='Annual income')] 

Yearly--->
[Document(id='AMT_INCOME_TOTAL', metadata={'id': 'AMT_INCOME_TOTAL'}, page_content='Annual income')] 

Database--->
[Document(id='OCCUPATION_TYPE', metadata={'id': 'OCCUPATION_TYPE'}, page_con

{'DAYS_BIRTH': 'Birthday',
 'OCCUPATION_TYPE': 'Occupation',
 'NAME_EDUCATION_TYPE': 'Education level',
 'CNT_FAM_MEMBERS': 'Family size',
 'AMT_INCOME_TOTAL': 'Annual income',
 'NAME_INCOME_TYPE': 'Income category'}

In [12]:
prompt=""" 
            You are a smart AI assistant, you will check if  the {description} can be used as one of the variables directly or indirectly to answer user question : {question} 
            Return the result in the format that matches this schema:\n{format_instructions}

            """
prompt_template=ChatPromptTemplate.from_template(prompt)

llm_model_with_prompt=prompt_template|llm_model|Output_parser

retrivedDocs=retreiver.invoke("Which education has highest average income?")
print(retrivedDocs)
question=" I want to analyze the number of millenial customers who have alteast master degree with family of 4 people and earns more than 100K yearly"
# question="How customers are there under 18"

potentialColumns={
    "Column Name":list(),
    "Column Description":list(),
    "Usable" :list(),
    "Reason":list()
}
for col_name,col_description in unique_documents.items():
    print(col_name)
    result=llm_model_with_prompt.invoke({
                                    'question':question,
                                    'description':col_description,
                                    'format_instructions':Output_parser.get_format_instructions()
                                    }
                                    )
    print(result)
    potentialColumns["Column Name"].append(col_name)
    potentialColumns["Column Description"].append(col_description)
    potentialColumns["Usable"].append(result.Valid)
    potentialColumns["Reason"].append(result.Reason)

df=pd.DataFrame(potentialColumns)

[Document(id='NAME_EDUCATION_TYPE', metadata={'id': 'NAME_EDUCATION_TYPE'}, page_content='Education level')]
DAYS_BIRTH
Valid='yes' Reason='The birthday can be indirectly used to answer the question by calculating the age of the customer, which can be used to filter millennial customers.'
OCCUPATION_TYPE
Valid='yes' Reason="Occupation can be used as a variable indirectly to answer the question because certain occupations tend to have higher salaries and are more likely to have a master's degree."
NAME_EDUCATION_TYPE
Valid='yes' Reason="The education level (at least a master's degree) can be used directly to answer the question as it is a specific criterion mentioned in the question."
CNT_FAM_MEMBERS
Valid='yes' Reason="Family size can be used indirectly to filter customers who meet the income criterion, but it cannot be used directly to answer the question about the number of millennial customers with a Master's degree."
AMT_INCOME_TOTAL
Valid='yes' Reason='The annual income is a direc

In [13]:
df[df['Usable']=='yes']['Column Description'].values.tolist()


['Birthday',
 'Occupation',
 'Education level',
 'Family size',
 'Annual income',
 'Income category']

In [ ]:
print(str(df[df['Usable']=='yes']['Column Description']))

0         Birthday
2      Family size
3    Annual income
Name: Column Description, dtype: object


In [ ]:
datapoints=",".join(df[df['Usable']=='yes']['Column Description'].values.tolist())

In [ ]:
datapoints

'Birthday,Family size,Annual income'

In [48]:
class analysisTechniqueGenerator(BaseModel):
    analysis :list[str]=Field(description="Generate 3 analytics techniques names that needs to be used solve the question")

analysis_parser=PydanticOutputParser(pydantic_object=analysisTechniqueGenerator)

In [57]:
analysis_prompt=""" 
            You are a smart AI assistant with deep analytics knowledge in Banking industry and knowledge in different analytics techniques
            you will generate 3 analytics techniques names only which should be used to answer the question : {question}
            by using the following data points: {datapoints} 
            if you are unable to generate the analysis  , just state the reason for not generating 
            Return the result in the format that matches this schema:\n{format_instructions}

            """
prompt_template=ChatPromptTemplate.from_template(analysis_prompt)

llm_model_with_prompt=prompt_template|llm_model|analysis_parser

# question=" I want to analyze the number of millenial customers who have alteast masters degree with family of 4 people and earns more than 100K yearly"
question="How customers are there under 18"

datapoints=",".join(df[df['Usable']=='yes']['Column Description'].values.tolist())
result=llm_model_with_prompt.invoke({
                                    'question':question,
                                    'datapoints':datapoints,
                                    'format_instructions':analysis_parser.get_format_instructions()
                                    }
                                    )

print(result)


analysis=['Regression Analysis', 'Cluster Analysis', 'Segmentation Analysis']


In [53]:
df

,Column Name,Column Description,Usable,Reason
0,DAYS_BIRTH,Birthday,yes,The birthday can be indirectly used to answer ...
1,OCCUPATION_TYPE,Occupation,yes,Occupation can be used as a variable indirectl...
2,NAME_EDUCATION_TYPE,Education level,yes,The education level (at least a master's degre...
3,CNT_FAM_MEMBERS,Family size,yes,Family size can be used indirectly to filter c...
4,AMT_INCOME_TOTAL,Annual income,yes,The annual income is a direct requirement for ...
5,NAME_INCOME_TYPE,Income category,yes,The 'Income' category can be used directly to ...


In [12]:
df[df['Usable']=='yes']

,Column Name,Column Description,Usable,Reason
0,DAYS_BIRTH,Birthday,yes,The birthday can be used to calculate the age ...
2,CNT_FAM_MEMBERS,Family size,yes,Family size can be used indirectly to answer t...
3,AMT_INCOME_TOTAL,Annual income,yes,Annual income can be used indirectly to answer...


In [17]:
prompt=""" 
            You are a smart AI assistant, you will use the {columns} and generate SQL query to answer : {question} 
            Return the result in the format that matches this schema:\n{format_instructions}

            """
prompt_template=ChatPromptTemplate.from_template(prompt)

llm_model_with_prompt=prompt_template|llm_model|sqlQuery_parser

question=" I want to analyze the number of millenial customers who have alteast master degree with family of 4 people and earns more than 100K yearly"
columns=','.join(a for a,b in unique_documents.items())
print(columns)

result=llm_model_with_prompt.invoke({
                                    'question':question,
                                    'columns':columns,
                                    'format_instructions':sqlQuery_parser.get_format_instructions()
                                    }
                                    )
print(result)



DAYS_BIRTH,NAME_EDUCATION_TYPE,CNT_FAM_MEMBERS,AMT_INCOME_TOTAL


OutputParserException: Invalid json output: To generate the SQL query and return it in the specified format, I'll use the provided columns and conditions to create a query. 

Here's the SQL query:

```sql
SELECT 
  CONCAT('SELECT * FROM customers WHERE ', 
         'DAYS_BIRTH BETWEEN ', 
         (SELECT MIN(DAYS_BIRTH) FROM customers), 
         ' AND ', 
         (SELECT MAX(DAYS_BIRTH) FROM customers), 
         ' AND NAME_EDUCATION_TYPE = ''Master'' AND CNT_FAM_MEMBERS = 4 AND AMT_INCOME_TOTAL > 100000')
  AS query
```

However, this query will return the query for all millennial customers with the specified conditions. If you want to generate the query for millennial customers, you should add a condition to filter by birth year. 

Here's how you can do it:

```sql
SELECT 
  CONCAT('SELECT * FROM customers WHERE ', 
         'DAYS_BIRTH BETWEEN ', 
         (SELECT MIN(DAYS_BIRTH) FROM customers WHERE YEAR(CURRENT_DATE) - YEAR(DAYS_BIRTH) BETWEEN 25 AND 40), 
         ' AND ', 
  